# Get Processed L1 SLC Scene List from an L0 Scene List

Given a list of Sentinel-1 EW L0 scenes, looks up each one's tracking record in S3
(`S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER`) and writes the corresponding L1 SLC product names to
a text file. Scenes with no tracking record yet (not processed, still processing, or failed)
are reported separately rather than raising.

Simplified, list-driven version of the already-processed lookup done in section 3a of
[`02_batch_order_l0_to_slc_and_upload.ipynb`](./02_batch_order_l0_to_slc_and_upload.ipynb) --
that notebook lists the *entire* tracking folder to find already-processed scenes among all
`L0_PRODUCTS`; this one looks up a specific, known list of L0 scenes directly by key.

**Steps:** configure → check AWS credentials → look up each scene's tracking record → write the
L1 scene list.

## Configuration

Set the L0 scene list, output path, and S3 tracking location below.

In [ ]:
import os
import json
from dotenv import load_dotenv

try:
    # load credentials from root .env file
    load_dotenv("../../.env")
except:
    print("Could not find .env file with credentials.")

# --- INPUT -------------------------------------------------------------------
# L0 scenes to look up. Point at a text file (one scene name per line) or set directly,
# e.g.:
# L0_PRODUCTS = ["S1A_EW_RAW__0SSH_20220131T034822_20220131T034930_041699_04F623_CCA2.SAFE"]
L0_PRODUCTS = [line.strip() for line in open("EW_L0_shackleton_BOM_scene_list_mar26.txt") if line.strip()]

# Where to write the resulting L1 SLC scene name list
OUTPUT_LIST_PATH = "EW_L1_shackleton_BOM_scene_list_mar26.txt"

# If True, write each entry as a full S3 URL, e.g.:
# https://deant-data-public-dev.s3.ap-southeast-2.amazonaws.com/s1_ew_slc/data/S1A_EW_SLC__1SDH_20250103T152034_20250103T152140_057281_070C14_5AA3.SAFE.zip
# If False, write the bare scene name (no .zip), e.g. S1A_EW_SLC__1SDH_..._5AA3.SAFE
WRITE_SLC_AS_URL_PATH = True

# S3 location of the tracking records, keyed by L0 scene name (<L0 basename>.json), and
# of the uploaded SLC products themselves (used only when WRITE_SLC_AS_URL_PATH is True)
S3_BUCKET = "deant-data-public-dev"
S3_PROJECT_FOLDER = "s1_ew_slc"
S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/tracking/L0_RAW"
S3_SCENE_UPLOAD_FOLDER = f"{S3_PROJECT_FOLDER}/data"
AWS_REGION = "ap-southeast-2"

# AWS profile to use for reading S3. If an expired SSO session is hit, run `aws sso login
# --profile <profile>` manually and re-run the credential check cell below.
AWS_PROFILE_OVERRIDE = ""  # e.g. "GASuperDeveloper-451924316694"
# -----------------------------------------------------------------------------

print(f"L0 scenes  : {len(L0_PRODUCTS)}")
print(f"Output path: {os.path.abspath(OUTPUT_LIST_PATH)}")
print(f"Output form: {'S3 URL' if WRITE_SLC_AS_URL_PATH else 'scene name'}")

## 1. AWS Credentials

Confirms credentials are resolvable and identifies the caller. Re-run this cell if it fails
with `TokenRetrievalError`/`UnauthorizedSSOTokenError` after running
`aws sso login --profile <profile>`.

In [ ]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError, TokenRetrievalError, UnauthorizedSSOTokenError

profile = AWS_PROFILE_OVERRIDE or os.environ.get("AWS_PROFILE")
if profile:
    print(f"Using AWS profile '{profile}'.")

try:
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    identity = session.client("sts").get_caller_identity()
    print(f"AWS credentials OK: {identity['Arn']}")
except (TokenRetrievalError, UnauthorizedSSOTokenError) as e:
    raise RuntimeError(
        f"SSO session for profile '{profile}' has expired. Run:\n\n"
        f"    aws sso login --profile {profile}\n\nthen re-run this cell."
    ) from e
except NoCredentialsError:
    raise RuntimeError("No AWS credentials found.")

s3_client = session.client("s3")

## 2. Look Up Tracking Records

For each L0 scene, fetches its tracking record directly by key
(`<S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER>/<L0 basename>.json`) and reads its `l1_product`
field -- no need to list the whole tracking folder.

In [ ]:
def get_l1_product_name(l0_scene: str) -> str | None:
    """Return the L1 SLC product name (as stored, including .zip) from an L0 scene's
    tracking record, or None if the scene has no tracking record yet (not processed,
    still processing, or failed)."""
    l0_basename = os.path.splitext(os.path.basename(l0_scene))[0]
    key = f"{S3_L0_RAW_SCENE_TRACKING_UPLOAD_FOLDER.rstrip('/')}/{l0_basename}.json"
    try:
        record = json.loads(s3_client.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read())
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchKey":
            return None
        raise
    return record.get("l1_product")


l1_scenes = []
unprocessed_scenes = []
for l0_scene in L0_PRODUCTS:
    l1_product = get_l1_product_name(l0_scene)
    if l1_product:
        l1_scenes.append(l1_product)
    else:
        unprocessed_scenes.append(l0_scene)

print(f"Processed    : {len(l1_scenes)}/{len(L0_PRODUCTS)}")
print(f"Not processed: {len(unprocessed_scenes)}")
for scene in unprocessed_scenes:
    print(f"  {scene}")

## 3. Write the L1 Scene List

Writes either bare scene names or full S3 URLs, per `WRITE_SLC_AS_URL_PATH`.

In [ ]:
if WRITE_SLC_AS_URL_PATH:
    entries = sorted(
        f"https://{S3_BUCKET}.s3.{AWS_REGION}.amazonaws.com/{S3_SCENE_UPLOAD_FOLDER.rstrip('/')}/{name}"
        for name in l1_scenes
    )
else:
    entries = sorted(os.path.splitext(name)[0] for name in l1_scenes)

with open(OUTPUT_LIST_PATH, "w") as f:
    f.write("\n".join(entries) + "\n")

print(f"Wrote {len(entries)} L1 scene entry(ies) to {os.path.abspath(OUTPUT_LIST_PATH)}")